In [3]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_csv('/content/Final_Marks_Data (1).csv')

In [5]:
df.head()

,Student_ID,Attendance (%),Internal Test 1 (out of 40),Internal Test 2 (out of 40),Assignment Score (out of 10),Daily Study Hours,Final Exam Marks (out of 100)
0,S1000,84,30,36,7,3,72
1,S1001,91,24,38,6,3,56
2,S1002,73,29,26,7,3,56
3,S1003,80,36,35,7,3,74
4,S1004,84,31,37,8,3,66


In [6]:
low = df["Final Exam Marks (out of 100)"].quantile(0.33)
high = df["Final Exam Marks (out of 100)"].quantile(0.66)

def categorize(x):
    if x < low:
        return "Low"
    elif x < high:
        return "Medium"
    else:
        return "High"

df["performance"] = df["Final Exam Marks (out of 100)"].apply(categorize)

In [7]:
df.head()

,Student_ID,Attendance (%),Internal Test 1 (out of 40),Internal Test 2 (out of 40),Assignment Score (out of 10),Daily Study Hours,Final Exam Marks (out of 100),performance
0,S1000,84,30,36,7,3,72,High
1,S1001,91,24,38,6,3,56,Low
2,S1002,73,29,26,7,3,56,Low
3,S1003,80,36,35,7,3,74,High
4,S1004,84,31,37,8,3,66,Medium


In [8]:
df[df['Final Exam Marks (out of 100)']>=70].head()

,Student_ID,Attendance (%),Internal Test 1 (out of 40),Internal Test 2 (out of 40),Assignment Score (out of 10),Daily Study Hours,Final Exam Marks (out of 100),performance
0,S1000,84,30,36,7,3,72,High
3,S1003,80,36,35,7,3,74,High
5,S1005,100,34,34,7,3,79,High
6,S1006,96,40,36,8,3,83,High
7,S1007,83,39,37,7,3,77,High


In [9]:
df.drop("Final Exam Marks (out of 100)", axis=1, inplace=True)
df.drop("Student_ID", axis=1, inplace=True)

In [10]:
df.shape

(2000, 6)

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   Attendance (%)                2000 non-null   int64 
 1   Internal Test 1 (out of 40)   2000 non-null   int64 
 2   Internal Test 2 (out of 40)   2000 non-null   int64 
 3   Assignment Score (out of 10)  2000 non-null   int64 
 4   Daily Study Hours             2000 non-null   int64 
 5   performance                   2000 non-null   object
dtypes: int64(5), object(1)
memory usage: 93.9+ KB


In [12]:

df.duplicated().sum()

np.int64(44)

In [13]:
df['performance'].value_counts()

,count
performance,
Medium,694
High,686
Low,620


In [14]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["performance"] = le.fit_transform(df["performance"])

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [16]:
X = df.drop("performance", axis=1)
y = df["performance"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier,StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

dt = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

svm = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    class_weight='balanced',
    probability=True,
    random_state=42
)

voting_class= VotingClassifier(
    estimators=[
        ('dt', dt),
        ('rf', rf),
        ('svm', svm)
    ],
    voting='hard'
)


stacking = StackingClassifier(
    estimators=[
        ('voting', voting_class)
    ],
    final_estimator=xgb
)
stacking.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:26:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


StackingClassifier(estimators=[('voting',
                                VotingClassifier(estimators=[('dt',
                                                              DecisionTreeClassifier(class_weight='balanced',
                                                                                     max_depth=5,
                                                                                     random_state=42)),
                                                             ('rf',
                                                              RandomForestClassifier(class_weight='balanced',
                                                                                     random_state=42)),
                                                             ('svm',
                                                              SVC(class_weight='balanced',
                                                                  probability=True,
                                                                  random_state=42))]))],
                   final_estimator=XGBClassifier(base_score=None, booste...
                                                 feature_weights=None,
                                                 gamma=None, grow_policy=None,
                                                 importance_type=None,
                                                 interaction_constraints=None,
                                                 learning_rate=0.1,
                                                 max_bin=None,
                                                 max_cat_threshold=None,
                                                 max_cat_to_onehot=None,
                                                 max_delta_step=None,
                                                 max_depth=4, max_leaves=None,
                                                 min_child_weight=None,
                                                 missing=nan,
                                                 monotone_constraints=None,
                                                 multi_strategy=None,
                                                 n_estimators=100, n_jobs=None,
                                                 num_parallel_tree=None, ...))

In [18]:
from sklearn.metrics import classification_report

y_pred = stacking.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.78      0.80       132
           1       0.87      0.74      0.80       139
           2       0.60      0.72      0.65       129

    accuracy                           0.75       400
   macro avg       0.76      0.75      0.75       400
weighted avg       0.76      0.75      0.75       400



In [19]:
df.head()

,Attendance (%),Internal Test 1 (out of 40),Internal Test 2 (out of 40),Assignment Score (out of 10),Daily Study Hours,performance
0,84,30,36,7,3,0
1,91,24,38,6,3,1
2,73,29,26,7,3,1
3,80,36,35,7,3,0
4,84,31,37,8,3,2


In [27]:
attendance = float(input("Enter Attendance (%): "))
it1 = float(input("Enter Internal Test 1 (out of 40): "))
it2 = float(input("Enter Internal Test 2 (out of 40): "))
assignment = float(input("Enter Assignment Score (out of 10): "))
study_hours = float(input("Enter Daily Study Hours: "))

input_data = np.array([[attendance, it1, it2, assignment, study_hours]])

prediction = stacking.predict(input_data)


original_label = le.inverse_transform(prediction)

print("Predicted Performance:", original_label[0])

Enter Attendance (%): 40
Enter Internal Test 1 (out of 40): 14
Enter Internal Test 2 (out of 40): 16
Enter Assignment Score (out of 10): 5
Enter Daily Study Hours: 3
Predicted Performance: Low


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


In [21]:
X_test

,Attendance (%),Internal Test 1 (out of 40),Internal Test 2 (out of 40),Assignment Score (out of 10),Daily Study Hours
1301,79,29,31,6,3
87,84,40,36,9,4
1451,82,28,34,6,3
599,90,32,32,7,4
872,81,30,33,7,3
...,...,...,...,...,...
667,84,33,36,7,4
1259,80,28,31,7,2
448,83,32,32,7,3
660,83,29,31,8,3


In [22]:
y_test

,performance
1301,1
87,0
1451,2
599,2
872,1
...,...
667,0
1259,1
448,1
660,2


In [25]:
le.classes_

array(['High', 'Low', 'Medium'], dtype=object)

In [26]:
from xgboost import XGBClassifier